# A dead ReLU is an infeasible stationary point of the Lagrangian

`counter.ipynb` shows a synthetic `(f, g)` pair on which ALM stalls: the iterate slides onto a
plateau where `f' = g' = 0` while `g > 0`, so the Lagrangian gradient is exactly zero for every
multiplier, the primal stops moving and the dual diverges. This notebook builds the same
phenomenon out of standard deep-learning parts.

**Mechanism.** Only two properties matter: an open set `P` with `grad f = grad g = 0` on it, and
`g > 0` on `P`. Then for any augmentation `psi`,

    grad L = grad f + psi'(lambda, rho, g) * grad g = 0    on P, for every (lambda, rho)

so every point of `P` is a stationary point of the augmented Lagrangian *simultaneously for all
duals*. Note what still holds: `||grad L|| -> 0`, and the primal iterate converges. Only
asymptotic feasibility fails.

This needs the absence of weak convexity. If `g` is convex, `0 in dg(t0)` makes `t0` a global
minimiser of `g`, so `g(t0) > 0` means the *problem* is infeasible. If `g` is `rho`-weakly convex,
`g(z) >= g(t0) - (rho/2)||z - t0||^2` forces every feasible `z` to satisfy
`||z - t0|| >= sqrt(2 g(t0) / rho)` — a trap can exist, but only that far away, and it recedes to
infinity as `rho` decreases. A trap adjacent to the feasible set needs `rho = infinity`, i.e. a
**concave kink**: a downward jump in slope, which no quadratic can convexify. The kinks at
`x = 2` and `x = 5` in `counter.ipynb` are exactly that.

**Where deep learning supplies one.** If `G(t) = h(relu(s(t)))` with `grad s != 0` and
`h'(0+) < 0`, then across `{s = 0}` the directional slope of `G` jumps from `0` to
`h'(0+) grad s` — downward. So `G` is not `rho`-weakly convex for any `rho`, and `{s < 0}` is a
flat plateau at value `h(0)`, infeasible whenever `h(0) > 0`.

`h'(0+) < 0` means *the constraint is relieved by making the unit more active*, which splits the
usual fairness constraints in two:

| | example | traps? |
|---|---|---|
| **level-style** | per-group loss cap, worst-group risk, minimum group recall / TPR / selection rate, coverage | **yes** — violated by the constant predictor |
| **gap-style** | DP gap, disparate-impact ratio, EO gap | no — *satisfied* by the constant predictor, so `h(0) <= 0` |

## The problem

One hidden ReLU unit, linear read-out, squared loss, cap on the minority group's risk:

    yhat(x; t) = v * relu(w x + b),   t = (w, b, v)
    f(t) = mean_i (yhat_i - y_i)^2
    g(t) = mean_{i in B} (yhat_i - y_i)^2 - 0.25 <= 0

Data: one feature with three design points. Group A (99%) sits at `x = 0` and `x = 2`, both with
`y = 0`; group B (1%) sits between them at `x = 1` with `y = 1`. The features *do* distinguish the
groups, but one ReLU unit still cannot serve both: fitting A requires `b <= 0` and `2w + b <= 0`,
which together force `w + b <= 0` and so switch B off — a single unit cannot make a bump.
Satisfying the constraint therefore costs majority accuracy: at the constrained optimum
`(w, b) = (0, 0.5)` the unit predicts `0.5` for *everyone*, giving `f = 0.25` against `f = 0.01`
unconstrained.

That last number is the one that matters here. The region where all three points are switched off
is a *local* minimiser set of `f` with value `0.01`, against a global minimum of `0.0099` at
`(w, b) = (0, 0.01)` just outside it — a 1% gap. So plain loss descent runs into the trap from a
natural initialisation and the loss barely notices, while constraint pressure is the only thing
that can hold the iterate out of it.

In [ ]:
import torch
from matplotlib import pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from humancompatible.train.dual_optim import ALM, PBM, SSG

# group A (99%): x = 0 and x = 2, both y = 0.   group B (1%): x = 1, y = 1.
XV, NV, YV = [0.0, 2.0, 1.0], [495, 495, 10], [0.0, 0.0, 1.0]
X = torch.cat([torch.full((n, 1), x) for x, n in zip(XV, NV)])
Y = torch.cat([torch.full((n, 1), y) for n, y in zip(NV, YV)])
IS_B = torch.zeros(sum(NV), dtype=torch.bool); IS_B[-NV[2]:] = True
X_B = X[IS_B][:1]                      # one group-B input, for reporting yhat
KAPPA = 0.25                           # group-B risk cap

INIT_alive = torch.tensor([0.1, 0.8, 1.0])   # all three points active, g = -0.24
INIT_dead = torch.tensor([-0.1, -0.1, 1.0])  # inside the dead cone


def predict(t, x, act=torch.relu):
    return t[2] * act(t[0] * x + t[1])


def objective(t, act=torch.relu):
    return ((predict(t, X, act) - Y) ** 2).mean()


def constraint(t, act=torch.relu):     # g <= 0
    yb = predict(t, X[IS_B], act)
    return (((yb - Y[IS_B]) ** 2).mean() - KAPPA).reshape(1)


for name, t in [("dead cone   (0, -1)", [0.0, -1.0, 1.0]),
                ("dead cone   (1, -3)", [1.0, -3.0, 1.0]),
                ("global min  (0, 0.01)", [0.0, 0.01, 1.0]),
                ("constrained opt (0, 0.5)", [0.0, 0.5, 1.0]),
                ("init", INIT_alive)]:
    t = t if torch.is_tensor(t) else torch.tensor(t)
    yh = [round(predict(t, torch.tensor([[x]])).item(), 3) for x in XV]
    print(f"{name:<27} f = {objective(t):.6f}  g = {constraint(t).item():+.4f}  yhat = {yh}")

print("\nthe dead cone {b <= 0, 2w + b <= 0} is a local minimiser set of f at 0.01, just "
      "1% above\nthe global minimum 0.0099 -- cheap enough that the loss curve never "
      "flags the trap.")

## The shape

On the slice `w = 0` the unit is constant across `x`, so all three design points share the single
pre-activation `b` and the problem collapses to the same pair of curves as `counter.ipynb`:

    F(b) = b^2 - 0.02 b + 0.01   (b >= 0),   0.01   (b < 0)
    G(b) = (b - 1)^2 - 0.25      (b >= 0),   0.75   (b < 0)

The constrained optimum `(w, b) = (0, 0.5)` lies on this slice: `f = 0.25`, `lambda* = 0.98`, and
KKT holds, so the problem is well posed. Both curves have a concave kink at `b = 0` (`F'`:
`0 -> -0.02`, `G'`: `0 -> -2`) and a flat, infeasible plateau on `b < 0`. Off this slice neither
function is a function of one variable any more — hence the 2D picture further down.

In [ ]:
def slice_1d(bs):
    # f and g along the w = 0 slice, where all three design points share relu(b)
    f, g = [], []
    for b in bs:
        t = torch.tensor([0.0, float(b), 1.0])
        f.append(objective(t).item()); g.append(constraint(t).item())
    return f, g


b_slice = torch.linspace(-1.0, 2.0, 601)
F, G = slice_1d(b_slice)

fig, ax = plt.subplots(figsize=(7, 4))
ax.axvspan(0.5, 1.5, color="tab:green", alpha=0.12, label="feasible set")
# ax.axvspan(-1.0, 0.0, color="tab:red", alpha=0.12, label="dead cell (flat, infeasible)")
ax.plot(b_slice, F, label="f(0, b)")
ax.plot(b_slice, G, label="g(0, b)")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", ls="--", lw=0.8)
ax.set_ylim(-0.45, 1.7)
ax.set_xlabel("bias b   (slice w = 0)")
ax.set_ylabel("value")
ax.legend(fontsize=8)
plt.tight_layout()

## The stall

One SGD step on the augmented Lagrangian. At the start `(w, b) = (0.1, 0.8)` all three points are
active and `g = -0.24`: the constraint is **satisfied**, so `lambda = 0`, `[g]_+ = 0`, and the
augmented Lagrangian is *exactly* `f` — the dual has no information yet. There
`grad f = (1.978, 1.780)`, so a step of size `eta` sends `w -> 0.1 - 1.978 eta` and
`b -> 0.8 - 1.780 eta`, and each design point switches off at its own threshold:

    x = 2   (2w + b <= 0):   eta >= 0.174
    x = 1   (w + b <= 0):    eta >= 0.240    <- grad g = 0: the constraint goes blind
    x = 0   (b <= 0):        eta >= 0.449    <- grad f = 0 as well: the full trap

Between the second and third threshold the run is in a partial state: the dual is already
powerless, while `grad f != 0` keeps moving the iterate — and because `f`'s minimiser set *is*
the dead cone, the objective finishes the descent into it unaided. The sweep below shows both
routes.

Below we plot the `v`-frozen model.

In [ ]:
def run(make_dual, eta, iters=400, act=torch.relu, init=INIT_alive, train_v=True,
        lam0=None, ramp=1.0, ramp_every=10, reinit_dead=False):
    # one primal-dual run; returns a per-iteration log
    t = torch.nn.Parameter(init.clone())
    opt = torch.optim.SGD([t], lr=eta)
    dual = make_dual()
    if lam0 is not None:
        dual.param_groups[0]["params"][0].fill_(lam0)
    log = []
    for k in range(iters):
        if ramp != 1.0 and k > 0 and k % ramp_every == 0:
            dual.penalty *= ramp                       # the usual ALM penalty escalation
        f, c = objective(t, act), constraint(t, act)
        L = dual.forward_update(f, c)
        opt.zero_grad()
        L.backward()
        if not train_v:
            t.grad[2] = 0.0
        log.append(dict(w=t[0].item(), b=t[1].item(),
                        s=(t[0] + t[1]).item(), yhat=predict(t, X_B, act).item(),
                        f=f.item(), g=c.item(), lam=dual.duals.max().item(),
                        rho=getattr(dual, "penalty", float("nan")),
                        gn=t.grad.norm().item()))
        opt.step()
        if reinit_dead and (t[0] + t[1]) <= 0:         # resurrect the dead unit
            with torch.no_grad():
                t.copy_(INIT_alive)
    return log


def trapped(log):
    # stalled at an infeasible point: the gradient has vanished but g > 0.  The
    # iterate reaches the cone's interior (gradient exactly 0) for large eta and
    # its boundary (gradient -> 0 geometrically) for moderate eta, so testing for
    # exactly 0.0 would miss the second case.
    return log[-1]["gn"] < 1e-8 and log[-1]["g"] > 1e-6


def verdict(log):
    z = log[-1]
    if not (z["gn"] == z["gn"] and abs(z["gn"]) < float("inf")):
        return "DIVERGED"
    return "TRAPPED" if trapped(log) else "ok"


def show(name, log):
    z = log[-1]
    print(f"{name:<26} {verdict(log):<8} "
          f"w+b {log[0]['s']:+.3f} -> {z['s']:+.4f} | yhat_B {z['yhat']:+.4f} | f {z['f']:.5f} | "
          f"g {z['g']:+.5f} | lam {z['lam']:.4f} | |grad L| {z['gn']:.2e}"
          f"{' (exactly 0)' if z['gn'] == 0.0 else ''}")


def alm(lr=0.1, penalty=1.0, **kw):
    return lambda: ALM(m=1, lr=lr, penalty=penalty, is_ineq=True, **kw)

runs = {eta: run(alm(), eta, train_v=False) for eta in (0.10, 0.20, 0.30)}
for eta, log in runs.items():
    show(f"eta = {eta}", log)

print()
for i, r in enumerate(run(alm(), 0.30, iters=8, train_v=False)):
    print(f"  it{i}: w={r['w']:+.5f} b={r['b']:+.5f}  f={r['f']:.6f}  "
          f"g={r['g']:+.6f}  lam={r['lam']:.4f}  |grad L|={r['gn']:.3e}")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 3.4))
for key, ax, title in [("f", axs[0], "objective f"),
                       ("g", axs[1], "constraint value g"),
                       ("lam", axs[2], "multiplier"),
                       ("gn", axs[3], "||grad L||")]:
    for eta, c in [(0.10, "tab:blue"), (0.20, "tab:green"), (0.30, "tab:red")]:
        ax.plot([r[key] for r in runs[eta]], color=c, label=f"eta={eta}")
    ax.set_xlabel("iteration"); ax.set_title(title); ax.legend(fontsize=7)
axs[1].axhline(0, color="k", lw=0.5)
axs[3].set_yscale("symlog", linthresh=1e-12)
plt.tight_layout()

## The same picture in parameter space

With three design points the network is piecewise linear over three kink lines in `(w, b)` —
`b = 0`, `2w + b = 0` and `w + b = 0`, one per design point — and the level sets are genuinely
two-dimensional. Fixing `v = 1`:

* the **dead cone** `{b <= 0, 2w + b <= 0}` is an open region on which all three points are off,
  so `f = 0.01` and `g = 0.75` are constant: it is a single level set of *both* functions at once,
  and simultaneously the objective's global minimiser set;
* the **feasible region** `{0.5 <= w + b <= 1.5}` is the strip between the two black lines, with
  the constrained optimum `(0, 0.5)` on its lower edge;
* crossing `w + b = 0` alone already sets `grad g = 0`, so the dual goes blind on a strictly
  larger region than the cone.

The third panel is the one that matters: it is a level-set plot of the augmented Lagrangian's
gradient norm at the optimal multiplier. The cone is not a region of *small* gradient — it is the
zero level set, exactly, and the same holds for every other `(lambda, rho)`.

In [ ]:
LAM_STAR, RHO = 0.98, 1.0                 # the optimal multiplier, and ALM's penalty

w_grid = torch.linspace(-1.5, 1.5, 481)
b_grid = torch.linspace(-1.5, 2.0, 481)
W, B = torch.meshgrid(w_grid, b_grid, indexing="xy")


def grid_fg(W, B, v=1.0):
    # f and g on the (w, b) grid at fixed v, from the three design points
    yh = [v * torch.relu(W * x + B) for x in XV]
    n = sum(NV)
    f = sum((k / n) * (y - t) ** 2 for y, k, t in zip(yh, NV, YV))
    g = (yh[2] - YV[2]) ** 2 - KAPPA         # index 2 is group B (x = 1)
    return f, g


# the closed form above must agree with the functions the runs actually used
for probe in ([0.7, -0.2, 1.0], [0.2, -0.9, 1.0], [-1.3, 0.4, 1.0], [0.0, 0.5, 1.0]):
    t = torch.tensor(probe)
    fg = grid_fg(torch.tensor(probe[0]), torch.tensor(probe[1]))
    assert torch.allclose(fg[0], objective(t), atol=1e-6), probe
    assert torch.allclose(fg[1], constraint(t)[0], atol=1e-6), probe

Fg, Gg = grid_fg(W, B)

# ||grad_(w,b) L|| by autograd: each grid point's L depends only on its own (w, b),
# so one backward pass on the sum gives the whole field
Wg, Bg = W.clone().requires_grad_(True), B.clone().requires_grad_(True)
f_, g_ = grid_fg(Wg, Bg)
(f_ + LAM_STAR * g_ + 0.5 * RHO * g_.clamp(min=0) ** 2).sum().backward()
Lgrad = torch.sqrt(Wg.grad ** 2 + Bg.grad ** 2)

DEAD = (B <= 0) & (2 * W + B <= 0)        # both group-A points off => group B off too

fig, axs = plt.subplots(1, 3, figsize=(16.5, 4.8))
panels = [
    (axs[0], Fg, "f(w, b)", dict(levels=torch.linspace(0, 3, 25), cmap="viridis")),
    (axs[1], Gg, "g(w, b)   (red = infeasible)",
     dict(levels=torch.linspace(-0.25, 1.25, 25), cmap="RdBu_r",
          norm=TwoSlopeNorm(vcenter=0.0, vmin=-0.25, vmax=1.25))),
    (axs[2], Lgrad, f"||grad L(w, b)||   at lambda = {LAM_STAR}, rho = {RHO}",
     dict(levels=torch.linspace(0, 8, 25), cmap="viridis")),
]

for ax, Z, title, kw in panels:
    cf = ax.contourf(W, B, Z, extend="both", **kw)
    fig.colorbar(cf, ax=ax, fraction=0.046)

    # the dead cone, hatched on top so a flat region cannot be mistaken for a
    # merely small one
    cone_top = torch.minimum(torch.zeros_like(w_grid), -2 * w_grid)
    ax.fill_between(w_grid, b_grid[0].item(), cone_top, facecolor="none", hatch="///",
                    edgecolor="w", linewidth=0.0, zorder=3)
    for c in (0.0, 1.0, 2.0):             # the three ReLU kink lines, one per design point
        ax.plot(w_grid, -c * w_grid, "w--", lw=1.0, zorder=4,
                label="ReLU kinks" if c == 0.0 else None)
    ax.plot(w_grid, 0.5 - w_grid, "k-", lw=1.6, zorder=4, label="g = 0")
    ax.plot(w_grid, 1.5 - w_grid, "k-", lw=1.0, zorder=4)

    for eta, col in [(0.10, "tab:blue"), (0.20, "tab:green"), (0.30, "tab:red")]:
        log = runs[eta]
        ax.plot([r["w"] for r in log], [r["b"] for r in log], ".-", ms=3, lw=0.8,
                color=col, zorder=5, label=f"iterates, eta={eta}")
    ax.plot(*INIT_alive[:2], "w*", ms=14, mec="k", zorder=6, label="init")
    ax.plot(0.0, 0.5, "ko", ms=7, mfc="yellow", zorder=6, label="constrained opt")

    ax.set_xlim(w_grid[0], w_grid[-1]); ax.set_ylim(b_grid[0], b_grid[-1])
    ax.set_xlabel("w"); ax.set_ylabel("b"); ax.set_title(title, fontsize=10)

axs[0].legend(fontsize=7, loc="lower left")
axs[2].annotate("dead cone:  ||grad L|| = 0 exactly\n(for every lambda and rho)",
                xy=(-0.7, -0.6), xytext=(-1.4, 1.5), color="w", fontsize=8, zorder=7,
                arrowprops=dict(arrowstyle="->", color="w", lw=0.9))
plt.tight_layout()

print("max ||grad L|| over the dead cone {b <= 0, 2w + b <= 0}:", Lgrad[DEAD].max().item())
print("f on the dead cone:", Fg[DEAD].unique().tolist(),
      "  g on the dead cone:", Gg[DEAD].unique().tolist())

The stalled run has loss `0.01` — *better* than the constrained optimum's `0.25`, and within 1% of
the global unconstrained optimum `0.0099`, because the dead cone is a local minimiser set of the
objective. The loss curve looks perfect; only the frozen violation and the diverging multiplier
show the failure.
This is why dying ReLUs are harmless in unconstrained training and fatal here: feasibility can
require a *specific* unit to be alive, and no amount of dual pressure resurrects it.

The sweep below shows three regimes rather than a single threshold:

| `eta` | what happens |
|---|---|
| `<= 0.20` | converges to the constrained optimum, `g -> 0`, `lambda -> 0.98` |
| `0.25 – 0.40` | `grad g = 0` from iteration 1; the objective then drives `b -> 0+` geometrically, so the iterate converges to the cone's **boundary**: `g = 0.75` frozen, `f = 0.01`, `||grad L||` decaying to `1e-12` |
| `>= 0.45` | lands in the cone's **interior** at iteration 1, `||grad L|| = 0` exactly ever after |

The middle regime is worth noting: nothing is exactly zero in finite time, yet the run is just as
stuck, because the only surviving gradient component is the objective's and it points further into
the cone. With `v` frozen the geometric decay of `b` underflows to a hard zero in finite time
(iteration 100–316 below), so the exact and asymptotic cases are the same phenomenon.

In [ ]:
def blind_iter(log):
    # first iteration at which group B is off, so grad g = 0
    return next((i for i, r in enumerate(log) if r["w"] + r["b"] <= 0), None)


def death_iter(log):
    # first iteration after which the gradient is exactly zero forever
    return next((i for i in range(len(log)) if all(r["gn"] == 0.0 for r in log[i:])), None)


for train_v, label in [(True, "v trainable (3 parameters)"), (False, "v frozen (2 parameters)")]:
    print(f"{label} -- grad g = 0 at eta >= 0.240, grad f = 0 as well at eta >= 0.449")
    for eta in (0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.45, 0.50):
        log = run(alm(), eta, train_v=train_v)
        z = log[-1]
        print(f"  eta = {eta:<5} {'INFEASIBLE STALL' if trapped(log) else 'converged':<17} "
              f"blind@{str(blind_iter(log)):<5} exactly0@{str(death_iter(log)):<5} "
              f"|grad L|={z['gn']:.1e}  g={z['g']:+.4f}  f={z['f']:.5f}")
    print()

## The stall does not depend on the dual method or its hyperparameters

The primal direction of every method here lies in `span{grad f, grad g}`, so all of them see
exactly zero. SSG is the sharpest case: its convergence theory
([Huang & Lin, NeurIPS 2023](https://arxiv.org/abs/2301.13314)) assumes `f` and `c` are weakly
convex, and when the violation exceeds the tolerance it descends `max_i c_i` directly — whose
gradient is also zero here.

In [ ]:
show("ALM (quadratic)", run(alm(), 0.30))
show("ALM (hpr)", run(alm(augmentation="hpr"), 0.30))
show("SSG", run(lambda: SSG(m=1), 0.30))
show("PBM", run(lambda: PBM(m=1, gamma=0.1, gamma_annealing=False,
                            penalty_annealing=False), 0.30))

print("\nsame primal trajectory for any initial multiplier and any penalty:")
for lam0 in (0.0, 1.0, 10.0):
    for rho in (0.1, 1.0, 100.0):
        log = run(alm(penalty=rho), 0.30, lam0=lam0)
        print(f"  lam0={lam0:<5} rho={rho:<6} s_final={log[-1]['s']:+.6f} "
              f"g={log[-1]['g']:+.4f} |grad L|={log[-1]['gn']:.1e}")

## The escalating penalty walks into the trap on its own

The runs above needed a step size large enough to overshoot in one go. The failure does not: fix
`eta` at a value at which ALM converges to the exact optimum, start barely infeasible, and simply
let ALM escalate the penalty the way it normally does to become asymptotically exact.

The penalty term contributes curvature `~ rho * (dg/dyhat)^2` to the primal subproblem, so a
fixed step size violates the descent condition once `rho` is large enough. On a smooth or weakly
convex problem the result is oscillation and then divergence; here the overshoot lands in the dead
cone, where it **stops** — the trace below shows the last few oscillations growing, one step of
size `~300`, and then an exactly zero gradient for good. The *multiplier* is not the culprit:
right up to the death it tracks `lambda* = 0.98` and never grows large, and its later divergence
is a consequence of the trap rather than a cause. It is the penalty.

In [ ]:
INIT2 = torch.tensor([0.0, 0.45, 1.0])   # yhat = 0.45 everywhere, g = +0.0525
ETA = 0.10                               # ALM converges to the exact optimum here at rho = 1

# f on its own has no interest in feasibility: its descent path runs into the cone,
# a local minimum only 1% above f's global optimum
t = torch.nn.Parameter(INIT2.clone())
opt = torch.optim.SGD([t], lr=ETA)
for _ in range(400):
    opt.zero_grad(); objective(t).backward(); opt.step()
print(f"SGD on f alone at eta={ETA}: (w, b) = ({t[0].item():+.4f}, {t[1].item():+.4f}), "
      f"f = {objective(t).item():.5f}, g = {constraint(t).item():+.4f}   <- the dead cone\n")

print("with ALM, rho_0 = 1.0 escalated geometrically every 10 iterations:")
ramps = {}
for ramp in (1.0, 1.2, 1.5, 2.0):
    log = ramps[ramp] = run(alm(), ETA, init=INIT2, ramp=ramp)
    death = next((i for i in range(len(log)) if all(r["gn"] == 0.0 for r in log[i:])), None)
    print(f"  x{ramp:<5} {'TRAPPED' if trapped(log) else 'ok':<8} "
          f"death_iter={str(death):<6} rho_at_death="
          f"{log[death]['rho'] if death else log[-1]['rho']:.1f}   "
          f"yhat_final={log[-1]['yhat']:+.5f}  g_final={log[-1]['g']:+.4f}")

print("\nfixed penalty, no escalation:")
for rho in (1.0, 5.0, 10.0, 20.0, 50.0):
    log = run(alm(penalty=rho), ETA, init=INIT2)
    print(f"  rho={rho:<6} {'TRAPPED' if trapped(log) else 'ok':<8} "
          f"yhat_final={log[-1]['yhat']:+.5f}  g_final={log[-1]['g']:+.4f}")

In [ ]:
log = ramps[2.0]
fig, axs = plt.subplots(1, 3, figsize=(12, 3.4), sharex=True)
axs[0].plot([r["s"] for r in log]); axs[0].axhline(0, color="k", ls="--", lw=0.8)
axs[0].set_title("group-B pre-activation w + b\n(dies once the oscillation crosses 0)")
axs[1].plot([r["g"] for r in log]); axs[1].axhline(0, color="k", lw=0.5)
axs[1].set_title("constraint g")
axs[2].plot([r["rho"] for r in log], label="penalty rho")
axs[2].plot([r["lam"] for r in log], label="multiplier lambda")
axs[2].set_yscale("log"); axs[2].legend(fontsize=8)
axs[2].set_title("rho escalates by design;\nlambda only takes off after the trap")
for ax in axs:
    ax.set_xlabel("iteration")
plt.tight_layout()

d = next((i for i in range(len(log)) if all(r["gn"] == 0.0 for r in log[i:])), len(log) - 2)
print("oscillation across the kink, then absorption:")
for i in range(max(0, d - 7), min(len(log), d + 2)):
    r = log[i]
    print(f"  it{i:<3} rho={r['rho']:<8.1f} w={r['w']:+.5f} b={r['b']:+.5f} "
          f"yhat_B={r['yhat']:+.6f} g={r['g']:+.6f} lam={r['lam']:.4f} |grad L|={r['gn']:.2e}")

## What escapes, and what does not

Escape needs *non-locality*, not a better step size. The exact prox subproblem
`argmin_t {L(t) + ||t - t_k||^2 / 2 eta}` does leave the cone once `eta` exceeds a finite
threshold, because `L` is not globally constant — but a penalty-barrier method that solves that
subproblem with an inner first-order solver started at `t_k` does not, since the inner solver also
sees zero. Randomised smoothing or a Goldstein `eps`-subdifferential works iff `eps` exceeds the
distance to the kink. Reinitialising the dead unit leaves the absorbing set, but at a step size
that caused the death the run walks straight back in — a cycle between the initialisation and the
cone — so resurrection has to be paired with fixing the overshoot.

A smooth activation removes the exact zero: `softplus` converges to the constrained optimum at the
same step size at which `relu` stalls. A *leaky* one is not enough here — the leak lets `yhat` go
negative, which inflates `(yhat - 1)^2`, and at `eta = 0.30` the run diverges to `NaN` instead of
stalling. Removing the flat region converts the trap into either a slow escape or a blow-up,
depending on what the leak does to the constraint; only the smooth activation actually solves the
problem.

In [ ]:
leaky = lambda z: torch.nn.functional.leaky_relu(z, 0.01)
ETA_TRAP = 0.30                      # relu stalls here; see the sweep above

show("relu (reference)", run(alm(), ETA_TRAP))
show("leaky_relu(0.01)", run(alm(), ETA_TRAP, act=leaky))
show("softplus", run(alm(), ETA_TRAP, act=torch.nn.functional.softplus))
show("relu + reinit (cycles)", run(alm(), ETA_TRAP, reinit_dead=True))
show("relu, eta = 0.20", run(alm(), 0.20))

## Why this is not a strawman

- The dead cone `{b <= 0, 2w + b <= 0}` is an **open** subset of parameter space with positive
  measure, not a measure-zero kink — and it is a local minimiser set of the objective, only 1%
  above its global optimum, so loss descent runs into it and the loss barely notices. The strictly
  larger region `{w + b <= 0}` already has `grad g = 0`.
- On the cone the stochastic gradient is exactly zero with **zero variance** — this is not "small
  noisy gradients", and no SGD-escapes-saddles argument applies, since the cone is a flat local
  minimum of `L` rather than a strict saddle. Momentum decays to zero; Adam's update is
  `0/sqrt(0)`.
- In a wide network only the units active on the minority inputs need to die — few of them, since
  a minority group occupies a small region of input space. Alternatively a single saturated
  `clamp` / `hardtanh` / `relu6` head pins the group's predictions exactly, independently of any
  bias, and `max_pool` / top-k gives exactly zero gradient on the unselected paths.
- `sigmoid` gives the soft version, `grad ~ exp(-|z|)`, which underflows to *exactly* zero around
  `z = -90` in fp32 and `z = -17` in fp16/bf16 — so in low precision the soft trap becomes a hard
  one at very moderate logits.